In [41]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "src":
    BASE_DIR = BASE_DIR.parent

PROCESSED_DIR = BASE_DIR / "data" / "processed"

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

dia = pd.read_csv(
    PROCESSED_DIR / "dia_products.csv"
)

mercadona = pd.read_csv(
    PROCESSED_DIR / "mercadona_products.csv"
)

aldi = pd.read_csv(
    PROCESSED_DIR / "aldi_products.csv"
)
alcampo = pd.read_csv(
    PROCESSED_DIR / "alcampo_products.csv"
)
ahorramas = pd.read_csv(
    PROCESSED_DIR / "ahorramas_products.csv"
)

print("DIA:", len(dia))
print("Mercadona:", len(mercadona))
print("ALDI:", len(aldi))
print("Alcampo:", len(alcampo))
print("AhorraMás:", len(ahorramas))

for df in [dia, mercadona, aldi, alcampo, ahorramas]:
    if "supermarket" in df.columns:
        df["supermarket"] = df["supermarket"].astype(str).str.strip().str.title()
    if "searched_item" in df.columns:
        df["searched_item"] = df["searched_item"].astype(str).str.strip().str.lower()

BASE_DIR: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project
PROCESSED_DIR: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project\data\processed
DIA: 917
Mercadona: 895
ALDI: 787
Alcampo: 696
AhorraMás: 393


In [42]:
from normalize import normalize_units

dia = normalize_units(dia)
mercadona = normalize_units(mercadona)
aldi = normalize_units(aldi)
alcampo = normalize_units(alcampo)
ahorramas = normalize_units(ahorramas)

In [43]:
COMMON_COLUMNS = [
    "supermarket",
    "searched_item",
    "name",
    "price",
    "price_per_unit",
    "unit",
]
dia_compare = dia[COMMON_COLUMNS].copy()
mercadona_compare = mercadona[COMMON_COLUMNS].copy()
aldi_compare = aldi[COMMON_COLUMNS].copy()
alcampo_compare = alcampo[COMMON_COLUMNS].copy()
ahorramas_compare = ahorramas[COMMON_COLUMNS].copy()

In [44]:
all_products = pd.concat(
    [
        dia_compare,
        mercadona_compare,
        aldi_compare,
        alcampo_compare,
        ahorramas_compare,
    ],
    ignore_index=True
)

clean_products = all_products[
    (all_products["price_per_unit"] > 0.05) & 
    (all_products["price_per_unit"] <= 50.0)
].copy()


In [45]:
print(
    clean_products["supermarket"]
    .value_counts()
)

print("\nUnidades:")
print(
    clean_products["unit"]
    .value_counts()
)

print("\nBúsquedas:")
print(
    clean_products["searched_item"]
    .value_counts()
)

supermarket
Dia          895
Mercadona    880
Aldi         756
Alcampo      569
Ahorramás    231
Name: count, dtype: int64

Unidades:
unit
KILO        2232
LITRO        891
UNIDAD       150
LAVADO        35
DOCENA        11
ROLLO          4
CAPSULA        2
BOLSITA        2
METRO          2
SOBRE          1
PASTILLA       1
Name: count, dtype: Int64

Búsquedas:
searched_item
leche              607
queso              250
pan                222
pollo              215
pasta              196
aceite             172
huevos             166
agua               166
tomate             156
atún               140
zumo               139
arroz              137
yogur              137
carne picada       122
garbanzos          109
detergente         105
manzana             96
salmón              90
plátano             70
papel higiénico     36
Name: count, dtype: int64


In [46]:
leche = all_products[
    all_products["searched_item"] == "leche"
].copy()

display(
    leche[
        [
            "supermarket",
            "name",
            "price",
            "price_per_unit",
            "unit",
        ]
    ]
    .sort_values(
        ["unit", "price_per_unit"]
    )
)

,supermarket,name,price,price_per_unit,unit
1877,Aldi,Cápsulas de café con leche,3.17,0.20,CAPSULA
1879,Aldi,Cápsulas de café con leche descafeinado,3.17,0.20,CAPSULA
10,Dia,Huevos medianos M Dia 24 unidades,5.25,2.63,DOCENA
348,Dia,Huevos medianos M de gallinas criadas en suelo...,2.85,2.85,DOCENA
354,Dia,Huevos grandes L Dia 18 unidades,4.39,2.93,DOCENA
...,...,...,...,...,...
259,Dia,Cápsulas de cafe con leche Dolce Gusto 16 unid...,4.95,0.31,UNIDAD
271,Dia,Cápsulas de café con leche intenso Dolce Gusto...,4.99,0.31,UNIDAD
263,Dia,Cápsulas de café con leche descafeinado Dolce ...,5.19,0.32,UNIDAD
267,Dia,Cápsulas de café con leche Marcilla 16 unidades,5.89,0.37,UNIDAD


In [47]:
cheapest_by_supermarket = (
    all_products
    .sort_values("price_per_unit")
    .groupby(
        [
            "searched_item",
            "supermarket",
            "unit",
        ],
        as_index=False
    )
    .first()
)

display(cheapest_by_supermarket)

,searched_item,supermarket,unit,name,price,price_per_unit
0,aceite,Ahorramás,KILO,Mantequilla irlandesa Kerrygold 200g pastilla ...,22.45,112.250
1,aceite,Ahorramás,LITRO,Aceite de girasol Alipende garrafa 5l,1.72,0.344
2,aceite,Alcampo,LITRO,PRODUCTO ALCAMPO Aceite de girasol garrafa de ...,8.55,1.710
3,aceite,Aldi,KILO,Altramuces,1.45,3.820
4,aceite,Aldi,LITRO,Vinagre de vino blanco,0.62,0.620
...,...,...,...,...,...,...
171,zumo,Alcampo,LITRO,PRODUCTO ALCAMPO Bebida refrescante de zumo de...,1.40,0.700
172,zumo,Aldi,LITRO,Bebida Bahamas,1.19,0.600
173,zumo,Dia,LITRO,"Limonada Don Simón 1,5 L",1.35,0.900
174,zumo,Mercadona,KILO,Naranjas,5.85,1.950


In [48]:
cheapest_overall = (
    all_products
    .sort_values("price_per_unit")
    .groupby(
        [
            "searched_item",
            "unit",
        ],
        as_index=False
    )
    .first()
)

display(
    cheapest_overall[
        [
            "searched_item",
            "supermarket",
            "name",
            "price",
            "price_per_unit",
            "unit",
        ]
    ]
)

,searched_item,supermarket,name,price,price_per_unit,unit
0,aceite,Mercadona,Sal gruesa Hacendado,0.40,0.400,KILO
1,aceite,Ahorramás,Aceite de girasol Alipende garrafa 5l,1.72,0.344,LITRO
2,aceite,Mercadona,Tortas de aceite Inés Rosales,2.15,0.360,UNIDAD
3,agua,Dia,Aguacate bandeja 450 g,1.79,3.980,KILO
4,agua,Ahorramás,Agua San Joaquin 8L,0.12,0.015,LITRO
...,...,...,...,...,...,...
60,tomate,Mercadona,Pizzas tomate y queso Hacendado ultracongeladas,2.40,1.200,UNIDAD
61,yogur,Mercadona,Yogur sabores Hacendado,2.40,1.200,KILO
62,yogur,Mercadona,Salsa Yogur Hacendado,1.20,3.870,LITRO
63,zumo,Mercadona,Naranjas,5.85,1.950,KILO


In [49]:
display(
    all_products[
        all_products["searched_item"] == "leche"
    ][
        [
            "supermarket",
            "name",
            "price",
            "price_per_unit",
            "unit",
        ]
    ].head(50)
)

,supermarket,name,price,price_per_unit,unit
0,Dia,Leche semidesnatada Dia Láctea pack 6 x 1 L,5.04,0.84,LITRO
1,Dia,Leche entera Dia Láctea pack 6 x 1 L,5.76,0.96,LITRO
2,Dia,Leche semidesnatada sin lactosa Dia Láctea pac...,5.34,0.89,LITRO
3,Dia,Leche semidesnatada Dia Láctea 1 L,0.84,0.84,LITRO
4,Dia,Leche desnatada Dia Láctea pack 6 x 1 L,4.92,0.82,LITRO
5,Dia,Leche entera Dia Láctea 1 L,0.96,0.96,LITRO
6,Dia,Leche desnatada sin lactosa Dia Láctea pack 6 ...,5.10,0.85,LITRO
7,Dia,Leche semidesnatada sin lactosa Dia Láctea 1 L,0.89,0.89,LITRO
8,Dia,Leche entera Dia Láctea 3 x 200 ml,0.92,1.53,LITRO
9,Dia,Leche semidesnatada Asturiana pack 6 x 1 L,6.54,1.09,LITRO
